# Suite VLGDEF — Verilog definitions

`Definitions.define(unit)` gives the entities a source text declares, in its scopes, as IEEE 1800 resolves
names; `referents` tells what a name refers to.

In [ ]:
from mbse.Programs.Framework.Syntax import walk
from mbse.Programs.Verilog import Definitions as D, SystemVerilog2023, Syntax as S

SOURCES = "../../conformance/sources"


def define(text):
    unit = SystemVerilog2023.parse(text)
    return unit, D.define(unit)


def names(unit, spelling):
    return [n for n in walk(unit) if isinstance(n, S.NameExpression) and isinstance(n.name, S.Identifier)
            and n.name.spelling == spelling]

## VLGDEF-01 · Every entity of the conformance source, with its kind and qualified name

In [ ]:
unit = SystemVerilog2023.parse(open(f"{SOURCES}/systemverilog2023.sv", encoding="utf-8").read(), include_paths=[SOURCES])
program = D.define(unit)
entities = [str(e) for e in program.entities()]
assert entities[:6] == ["<package logger_pkg>", "<parameter logger_pkg::FIELDS>", "<localparam logger_pkg::BATTERY_LOW>",
                        "<enumerator logger_pkg::IDLE>", "<enumerator logger_pkg::SAMPLE>", "<enumerator logger_pkg::SEND>"]
assert "<argument logger_pkg::in_range::raw>" in entities and "<block sampler.counter>" in entities
assert "<localparam sampler.four.HALF>" in entities and "<instance sampler.u_fifo>" in entities
unresolved = sorted({n.name.spelling for n in walk(unit) if isinstance(n, S.NameExpression)
                     and isinstance(n.name, S.Identifier) and not D.referents(program, n)})
assert unresolved == ["tb"], unresolved
print(len(entities), "entities")

## VLGDEF-02 · Imports: explicit ones alias, wildcard ones are found where nothing nearer is

In [ ]:
unit, program = define("""package p; parameter A = 1; parameter B = 2; parameter C = 3; endpackage
package q; parameter C = 4; endpackage
module m; import p::A; import q::*; import p::missing; import nowhere::*; import nowhere::Y;
parameter B = 5;
initial x = A + B + C + p::B + nowhere::X + p::B2;
endmodule""")
a = names(unit, "A")[0]
assert [str(e) for e in D.referents(program, a)] == ["<parameter p::A>"]
assert [str(e) for e in D.referents(program, names(unit, "B")[0])] == ["<parameter m.B>"]
assert [str(e) for e in D.referents(program, names(unit, "C")[0])] == ["<parameter q::C>"]
scoped = [n for n in walk(unit) if isinstance(n, S.ScopedName)]
assert [str(e) for e in D.referents(program, scoped[0])] == ["<parameter p::B>"]
assert D.referents(program, scoped[1]) == [] and D.referents(program, scoped[2]) == []
missing = [e for e in program.entities() if e.name == "missing"][0]
assert missing.kind == "import" and missing.target is None
print("ok")

## VLGDEF-03 · Ports, parameters, types, enumerators, functions and blocks, each in its scope

In [ ]:
unit, program = define("""module m #(parameter type T = int, parameter N) (a, b);
input a; output b; typedef enum {X, Y} e_t;
genvar g;
function int f(int v); begin : inner int t; t = v; end return v; endfunction
initial begin int w; for (int i = 0; i < 2; i++) w = i; foreach (q[j]) w = j; for (;;) ; end
for (g = 0; g < 2; g++) begin : gen end
endmodule""")
entities = [str(e) for e in program.entities()]
assert entities[:7] == ["<module m>", "<type parameter m.T>", "<parameter m.N>", "<port m.a>", "<port m.b>", "<enumerator m.X>",
                        "<enumerator m.Y>"]
port = [e for e in program.entities() if e.name == "a"][0]
assert len(port.declarations) == 2 and isinstance(port.definition, S.PortReference)
assert "<function m.f>" in entities and "<argument m.f.v>" in entities and "<block m.f.inner>" in entities
assert "<variable m.f.inner.t>" in entities and "<block m.gen>" in entities and "<genvar m.g>" in entities
i = names(unit, "i")[0]
assert [e.kind for e in D.referents(program, i)] == ["variable"]
assert [e.kind for e in D.referents(program, names(unit, "j")[0])] == ["variable"]
assert D.referents(program, names(unit, "q")[0]) == []
declared = [n for n in walk(unit) if isinstance(n, S.Identifier) and n.spelling == "w"][0]
assert [str(e) for e in D.referents(program, declared)] == ["<variable w>"]
assert D.referents(program, S.Identifier(spelling="z")) == []
print("ok")

## VLGDEF-04 · Interfaces, modports, instances and out-of-block names

In [ ]:
unit, program = define("""interface bus; logic d; modport mp (input d); endinterface
module top; bus b(); sub u (.p(b.d)); endmodule
function int c::f(); endfunction""")
entities = [str(e) for e in program.entities()]
assert "<modport bus.mp>" in entities and "<instance top.b>" in entities and "<instance top.u>" in entities
assert not any("f" == e.name for e in program.entities() if e.kind == "function" and e.name is not None)
print("ok")